# M19d — Standard reservoir-task boundary panel

**Author:** Ildefons Magrans de Abril  
**Affiliation:** Universitat Politècnica de Catalunya - BarcelonaTech (UPC)

**Purpose.** Evaluate the temperature safe-region protocol on Lorenz-x, Mackey-Glass, delayed memory, and NARMA10.

**Provenance.** Task identities and historical summary statistics are recovered from the manuscript. The original notebook binary and exact historical seed schedule are unavailable. This publication notebook therefore performs a fresh protocol replication with the preserved later task implementations.

In [1]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import tcr_core as tcr

FROZEN = ROOT / "results" / "frozen"
REPRO = ROOT / "results" / "reproduced"
REPRO.mkdir(parents=True, exist_ok=True)

In [2]:
SEED=20260718
TRIALS=8
TASKS=["lorenz_x","mackey_glass","memory_d10","narma10"]
CONFIG=dict(N=60,K=13,lengths=(1200,500,500),washout=100,input_scale=0.8,ridge=1e-5)
hist=pd.read_csv(FROZEN/"historical_reference_metrics.csv")
display(hist[hist.notebook_id=="M19d"])

,notebook_id,metric,value,ci_low,ci_high,provenance
15,M19d,near_optimal_containment,0.968750,NaN,NaN,historical manuscript result
16,M19d,exact_oracle_containment,0.781250,NaN,NaN,historical manuscript result
17,M19d,safe_gain,0.004634,0.00016,0.009109,historical manuscript result


In [3]:
rep=tcr.run_panel(TASKS,TRIALS,["temperature"],seed=SEED,**CONFIG)
rep.to_csv(REPRO/"m19d_replication_case_metrics.csv",index=False)
summary=pd.DataFrame([{
    "n_cases":len(rep),"near_optimal_containment":rep.near_contained.mean(),
    "exact_oracle_containment":rep.exact_contained.mean(),"mean_safe_gain":rep.safe_gain.mean(),
    "mean_full_gain":rep.full_gain.mean(),"mean_safe_width":rep.safe_width.mean(),
}])
summary.to_csv(REPRO/"m19d_replication_summary.csv",index=False)
display(summary.round(6))
display(rep.groupby("task")[["safe_gain","full_gain","near_contained","exact_contained"]].mean().round(6))

,n_cases,near_optimal_containment,exact_oracle_containment,mean_safe_gain,mean_full_gain,mean_safe_width
0,32,0.84375,0.6875,0.001585,0.007261,2.71875


,safe_gain,full_gain,near_contained,exact_contained
task,,,,
lorenz_x,0.000922,0.005786,0.875,0.75
mackey_glass,0.001978,0.005544,0.875,0.50
memory_d10,0.001191,0.005975,0.875,0.75
narma10,0.002247,0.011739,0.750,0.75
